In [ ]:
##### MobileNet downloaden.
# https://download.pytorch.org/models/mobilenet_v2-b0353104.pth

In [ ]:
import os
import glob
from pathlib import Path

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image

# ==========================================
# 1. ZENTRALE KONFIGURATION (Config-Block)
# ==========================================

# Hardware-Erkennung (CUDA für Nvidia, MPS für Apple Silicon, sonst CPU)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Verzeichnisse & Download-Quelle für didaktische Zwecke
# DOWNLOAD-LINK FÜR MOBILENET V2: https://download.pytorch.org/models/mobilenet_v2-b0353104.pth
# Datei bitte herunterladen und im Ordner "Models" ablegen.
DATA_ROOT     = Path("Datasets")
TRAIN_DIR     = DATA_ROOT / "Train"
TEST_DIR      = DATA_ROOT / "Test"
MODEL_WEIGHTS = Path("Models/mobilenet_v2-b0353104.pth")

# Hyperparameter & Bildeigenschaften
IMG_SIZE   = 224
BATCH_SIZE = 32

# Hyperparameter: Transfer Learning (nur der neue Kopf)
EPOCHS_TRANSFER = 5
LR_TRANSFER     = 1e-3

# Hyperparameter: Fine-Tuning (Netzwerk teilweise auftauen)
EPOCHS_FINETUNE = 5
LR_HEAD         = 1e-4
LR_FEATURES     = 1e-5

# ImageNet-Statistiken (Identisch für fast alle vortrainierten PyTorch-Modelle)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Aktuelles Gerät: {DEVICE}")

In [ ]:
# ==========================================
# 2. DATEN LADEN & TRANSFORMIEREN
# ==========================================

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_ds = datasets.ImageFolder(str(TRAIN_DIR), transform=base_transform)
test_ds  = datasets.ImageFolder(str(TEST_DIR),  transform=base_transform)

print(f"Gefundene Klassen: {train_ds.classes}")
print(f"Anzahl Trainingsbilder: {len(train_ds)} | Testbilder: {len(test_ds)}")

use_accel = DEVICE.type in ["cuda", "mps"]
loader_kwargs = {
    "num_workers": 2 if use_accel else 0,
    "pin_memory": use_accel
}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **loader_kwargs)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False, **loader_kwargs)

In [ ]:
# ==========================================
# 3. MODELL LADEN (TRANSFER LEARNING)
# ==========================================

# Architektur laden (ohne Gewichte initialisieren)
mobilenet = models.mobilenet_v2(weights=None) 

# Sicherheitsprüfung für Lernende: Existiert die Datei?
if not MODEL_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Die Modellgewichte wurden nicht unter '{MODEL_WEIGHTS}' gefunden!\n"
        f"Bitte lade die Datei hier herunter: https://download.pytorch.org/models/mobilenet_v2-b0353104.pth "
        f"und erstelle den entsprechenden Ordner."
    )

# Heruntergeladene Gewichte manuell laden
state_dict = torch.load(MODEL_WEIGHTS, map_location="cpu")
mobilenet.load_state_dict(state_dict)

# 1. SCHRITT: Alte Features "einfrieren" (Kein Training der Faltungsschichten)
for param in mobilenet.features.parameters():
    param.requires_grad = False

# 2. SCHRITT: Neuen Klassifikator (Kopf) aufsetzen
# Bei MobileNet V2 liefert die Feature-Extraktion am Ende 1280 Kanäle.
NUM_CLASSES = len(train_ds.classes)        # ergibt sich aus den Ordnern (hier 4)
mobilenet.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(1280, NUM_CLASSES)           # ein Logit pro Klasse (Mehrklassen-Klassifikation)
)

model = mobilenet.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LR_TRANSFER)

# Hilfsfunktion zur Evaluierung
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)          # y bleibt long (Klassenindex)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total if total else 0.0

# 3. SCHRITT: Den neuen Kopf trainieren
print("--- Starte Transfer Learning (Nur der Kopf wird trainiert) ---")
for epoch in range(1, EPOCHS_TRANSFER + 1):
    model.train()
    running_loss, count = 0.0, 0
    
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)              # CrossEntropy erwartet long-Targets

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        count += x.size(0)

    train_loss = running_loss / count
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:02d}/{EPOCHS_TRANSFER} | Loss: {train_loss:.4f} | Test-Acc: {test_acc:.2f}%")

In [ ]:
# ==========================================
# 4. INFERENZ & PRODUKTIONS-EXPORT
# ==========================================

img_candidates = list(TEST_DIR.rglob("*.png")) + list(TEST_DIR.rglob("*.jpg"))
if img_candidates:
    img_path = img_candidates[0]
    
    im = Image.open(img_path).convert("RGB")
    x = base_transform(im).unsqueeze(0).to(DEVICE) 
    
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        pred_idx = int(probs.argmax(dim=1).item())
        conf = probs[0, pred_idx].item()
        
    print(f"Testbild: {img_path.name}")
    print(f"Vorhersage: {test_ds.classes[pred_idx]}")
    print(f"Konfidenz: {conf:.2%}")

# Optional: Als FP16 speichern
model.half()
torch.save(model, "Models/MobileNetV2_transferL_fp16_.pth")
print("\nModelle erfolgreich gespeichert.")

In [ ]:
# ==========================================
# 5. FINE-TUNING (Die letzten Schichten auftauen)
# ==========================================

# Falls Zelle 4 das Modell auf FP16 gesetzt hat: zurück auf FP32 fürs Training
# (Training in halber Präzision ist auf CPU/MPS instabil). Die bereits
# gespeicherte FP16-Datei bleibt davon unberührt.
model.float()

# MobileNet V2 besteht in model.features aus 19 Blöcken (Index 0 bis 18).
# Wir tauen die letzten beiden auf (Index 17: Inverted Residual, Index 18: Conv2dNormActivation).
for param in model.features[17:].parameters():
    param.requires_grad = True

# Wir verwenden unterschiedliche Lernraten: 
head_params    = [p for p in model.classifier.parameters() if p.requires_grad]
feature_params = [p for p in model.features[17:].parameters() if p.requires_grad]

optimizer_ft = optim.Adam([
    {"params": head_params,    "lr": LR_HEAD},
    {"params": feature_params, "lr": LR_FEATURES},
], weight_decay=1e-5)

print("\n--- Starte Fine-Tuning (Letzte Blöcke aufgetaut) ---")
for epoch in range(1, EPOCHS_FINETUNE + 1):
    model.train()
    running_loss, count = 0.0, 0
    
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer_ft.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer_ft.step()

        running_loss += loss.item() * x.size(0)
        count += x.size(0)

    train_loss = running_loss / count
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:02d}/{EPOCHS_FINETUNE} | Loss: {train_loss:.4f} | Test-Acc: {test_acc:.2f}%")

# Speichern des fertigen Modells
torch.save(model, "Models/MobileNetV2_finetuned_fp16_.pth")